# How to do the chunking

In [ ]:
from bs4 import BeautifulSoup
from typing import List
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

def get_heading_level(tag_name: str) -> int:
    """
    Returns the integer level of an HTML heading tag (e.g., 'h2' -> 2).
    Returns a high number for non-heading tags.
    """
    if tag_name and tag_name.startswith('h') and len(tag_name) == 2 and tag_name[1].isdigit():
        return int(tag_name[1])
    return float('inf')

def chunk_web_page(urls: List[str]) -> List[str]:
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.headless = True 
    
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_argument('--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')
    
    chrome_options.add_argument('--disable-images')
    chrome_options.add_argument('--disable-javascript') 
 
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    page_source = None
    driver = None
    results = []
    try:
        print("Initializing Chrome driver...")
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        
        driver.set_page_load_timeout(30)
        
        for url in urls:
            print(f"Navigating to URL: {url}")
            driver.get(url)
            
            time.sleep(5)
            
            print("Waiting for main content...")
            try:
                WebDriverWait(driver, 20).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "article, main, body"))
                )
            except Exception as wait_error:
                print(f"Warning: Timeout waiting for main content: {wait_error}")
            
            print("Getting page source...")
            page_source = driver.page_source
            chunks = process_page_content(page_source, url)
            results += chunks
            print(f"Successfully extracted {len(chunks)} chunks from {url}")

        
    except Exception as e:
        print(f"Error using Selenium to fetch URL {url}")
        print(f"Error type: {type(e).__name__}")
        print(f"Error details: {str(e)}")
        return []
    finally:
        if driver:
            print("Closing Chrome driver...")
            driver.quit()

    return results

def process_page_content(page_source: str, url: str) -> List[str]:
    """Extract chunks from page source and add page title to each chunk"""
    if not page_source:
        return []

    # Extract title from URL - get the last part after the last slash and replace hyphens
    page_title = url.split('/')[-1].replace('-', ' ').title()
    page_title = ' '.join(word for word in page_title.split() if not word[0].isdigit())
    title_prefix = f"Page Title: {page_title}\n\n"

    soup = BeautifulSoup(page_source, 'html.parser')
    main_content = soup.find('article') or soup.find('main') or soup.body

    # Remove irrelevant tags
    for tag in ['nav', 'header', 'footer', 'aside', 'script', 'style']:
        for s in main_content.find_all(tag):
            s.decompose()

    chunks = []
    headings = main_content.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])

    if not headings:
        # If no headings are found, fall back to chunking by paragraphs
        paragraphs = [p.get_text(strip=True) for p in main_content.find_all('p') if p.get_text(strip=True)]
        return [title_prefix + p for p in paragraphs]

    for heading in headings:
        current_level = get_heading_level(heading.name)
        chunk_content = [heading.get_text(strip=True)]
        
        for sibling in heading.find_next_siblings():
            sibling_level = get_heading_level(sibling.name)
            
            if sibling_level <= current_level:
                break

            text = sibling.get_text(separator=' ', strip=True)
            if text:
                chunk_content.append(text)
        
        if len(chunk_content) > 1:
            # Add the title prefix to each chunk
            chunks.append(title_prefix + '\n\n'.join(chunk_content))
            
    return chunks

In [16]:
target_urls = [
    "https://help.typeform.com/hc/en-us/articles/23541138531732-Create-multi-language-forms",
    "https://help.typeform.com/hc/en-us/articles/27703634781076-Add-a-Multi-Question-Page-to-your-form"]

results = chunk_web_page(target_urls)

Initializing Chrome driver...
Navigating to URL: https://help.typeform.com/hc/en-us/articles/23541138531732-Create-multi-language-forms
Waiting for main content...
Getting page source...
Successfully extracted 12 chunks from https://help.typeform.com/hc/en-us/articles/23541138531732-Create-multi-language-forms
Navigating to URL: https://help.typeform.com/hc/en-us/articles/27703634781076-Add-a-Multi-Question-Page-to-your-form
Waiting for main content...
Getting page source...
Successfully extracted 9 chunks from https://help.typeform.com/hc/en-us/articles/27703634781076-Add-a-Multi-Question-Page-to-your-form
Closing Chrome driver...


In [17]:
results


["Page Title: Create Multi Language Forms\n\nCreate multi-language forms\n\nSave time and reach a wider audience by creating a single form with multiple language options. You'll have the option to write your own translations or have AI translate your form for you.\xa0 Depending on your language settings, you can decide if you want to automatically have your form translated for respondents or give them the option to translate the form. The form will be translated if the respondent's browser language matches one of the translations you've added. All responses will be displayed in one place within your Results panel. You can create multi-language forms with the following languages: Arabic Catalan Chinese (simplified) Chinese (traditional) Croatian Danish Dutch English Estonian Finnish French German (formal) German (informal) Greek Hebrew Hungarian Italian Japanese Korean Norwegian Polish Portuguese Russian Spanish Swedish Turkish Ukrainian To create multi-language forms, you’ll need a Bus